<a href="https://colab.research.google.com/github/Soljafree60/git-work/blob/main/07_PINN_FDM_Strike_Sensitivity_CPUT_Harvard_SUBMISSION_READY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Strike-Price Sensitivity Experiment
## Analytical Black–Scholes vs Crank–Nicolson FDM vs Improved PINN

This notebook varies only the strike price:

$$
K \in \{80,\;100,\;120\}
$$

while holding

$$
S_0=100,\quad r=0.05,\quad \sigma=0.20,\quad T=1,\quad S_{\max}=200
$$

fixed.

For each strike value, the notebook computes the analytical benchmark, the $100\times100$ Crank–Nicolson solution, and a fresh improved PINN trained for 10,000 epochs with the staged learning-rate schedule and best-model checkpointing.

Both numerical methods are evaluated on the same 401-point asset-price grid.


## 1. Imports and reproducibility


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import time

from scipy.stats import norm
from scipy.linalg import solve_banded

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.keras.backend.set_floatx("float32")

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)


## 2. Parameters and strike scenarios


In [ ]:
S0 = 100.0
r = 0.05
sigma = 0.20
T = 1.0
S_max = 200.0

K_values = [80.0, 100.0, 120.0]

FDM_N = 100
FDM_M = 100

N_f = 10_000
N_b = 1_000
N_T = 1_000
EPOCHS = 10_000

lambda_f = 1.0
lambda_B = 1.0
lambda_T = 1.0

LR_1 = 1e-3
LR_2 = 5e-4
LR_3 = 1e-4

print("Strike values:", K_values)
print("FDM grid:", FDM_N, "x", FDM_M)
print("PINN epochs:", EPOCHS)


### Why I used these model parameters and hyperparameters

The Black–Scholes parameters define the financial problem being solved. The baseline values are held fixed. This produces a controlled reference case against which numerical changes can be measured (Black & Scholes, 1973; Hull, 2018).

The PINN hyperparameters define the computational budget and network training configuration. Keeping these settings explicit is important for reproducibility and makes it possible to distinguish changes in model accuracy caused by the market parameter from changes caused by the numerical procedure.


## 3. Analytical Black–Scholes solution


In [ ]:
def black_scholes_call(S, K, r, sigma, T, t=0.0):
    scalar_input = np.isscalar(S)
    S = np.asarray(S, dtype=float)
    tau = T - t

    if tau <= 0:
        result = np.maximum(S - K, 0.0)
        return float(result) if scalar_input else result

    result = np.zeros_like(S, dtype=float)
    positive = S > 0
    S_pos = S[positive]

    d1 = (
        np.log(S_pos / K)
        + (r + 0.5 * sigma**2) * tau
    ) / (sigma * np.sqrt(tau))

    d2 = d1 - sigma * np.sqrt(tau)

    result[positive] = (
        S_pos * norm.cdf(d1)
        - K * np.exp(-r * tau) * norm.cdf(d2)
    )

    return float(result) if scalar_input else result


### Why I used the analytical Black–Scholes function

The analytical European call solution is used as the benchmark because the Black–Scholes model has a closed-form solution under the assumptions used in this study (Black & Scholes, 1973; Hull, 2018). This allowed the numerical FDM and PINN solutions to be evaluated against a known reference rather than against one another.

The implementation handles $S=0$ and maturity separately to avoid undefined expressions such as $\log(0)$ or division by zero. These are numerical safeguards; they do not alter the Black–Scholes model.


## 4. Crank–Nicolson FDM solver


In [ ]:
def crank_nicolson_call(S_max, K, r, sigma, T, M, N):
    dt = T / M

    S = np.linspace(0.0, S_max, N + 1)
    V = np.maximum(S - K, 0.0)

    i = np.arange(1, N)

    a = 0.25 * dt * (sigma**2 * i**2 - r * i)
    b = -0.5 * dt * (sigma**2 * i**2 + r)
    c = 0.25 * dt * (sigma**2 * i**2 + r * i)

    A = np.zeros((3, N - 1))
    A[0, 1:] = -c[:-1]
    A[1, :] = 1.0 - b
    A[2, :-1] = -a[1:]

    for n in range(M):
        tau_new = (n + 1) * dt

        rhs = (
            a * V[:-2]
            + (1.0 + b) * V[1:-1]
            + c * V[2:]
        )

        V_left_new = 0.0
        V_right_new = S_max - K * np.exp(-r * tau_new)

        rhs[0] += a[0] * V_left_new
        rhs[-1] += c[-1] * V_right_new

        V_inner = solve_banded((1, 1), A, rhs)

        V[0] = V_left_new
        V[1:-1] = V_inner
        V[-1] = V_right_new

    return S, V


### Why the Crank–Nicolson method is used

Crank–Nicolson is an implicit finite-difference method obtained by averaging the spatial differential operator between two adjacent time levels. It is widely used for parabolic PDEs and option-pricing problems because it provides a strong balance between accuracy and numerical stability (Crank & Nicolson, 1947; Duffy, 2006).

The European call payoff is used as the initial condition in time-to-maturity coordinates, and the Black–Scholes boundary conditions are imposed at $S=0$ and $S=S_{\max}$.


## 5. PINN scaling and architecture


In [ ]:
def scale_inputs(S, t):
    S_scaled = 2.0 * S / S_max - 1.0
    t_scaled = 2.0 * t / T - 1.0
    return S_scaled, t_scaled


def model_prediction(model, S, t):
    S_scaled, t_scaled = scale_inputs(S, t)
    inputs = tf.concat([S_scaled, t_scaled], axis=1)
    return model(inputs)


class PINN(tf.keras.Model):
    def __init__(self):
        super().__init__()

        self.hidden1 = tf.keras.layers.Dense(
            64, activation="tanh",
            kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED + 1)
        )
        self.hidden2 = tf.keras.layers.Dense(
            64, activation="tanh",
            kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED + 2)
        )
        self.hidden3 = tf.keras.layers.Dense(
            64, activation="tanh",
            kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED + 3)
        )
        self.hidden4 = tf.keras.layers.Dense(
            64, activation="tanh",
            kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED + 4)
        )
        self.output_layer = tf.keras.layers.Dense(
            1,
            kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED + 5)
        )

    def call(self, inputs):
        x = self.hidden1(inputs)
        x = self.hidden2(x)
        x = self.hidden3(x)
        x = self.hidden4(x)
        return self.output_layer(x)


### Why I scaled the inputs

The asset price and time variables have different numerical ranges. Scaling them to approximately $[-1,1]$ improves the conditioning of the neural-network optimisation and is compatible with the use of `tanh` activation functions. PINNs are sensitive to optimisation and gradient behaviour, so numerical scaling can materially improve training stability (Karniadakis et al., 2021; Wang et al., 2021).

The PDE derivatives are still taken with respect to the physical variables $S$ and $t$. TensorFlow applies the chain rule through the scaling transformation automatically.


## 6. Shared collocation locations


In [ ]:
rng = np.random.default_rng(SEED)

S_f_np = rng.uniform(0.0, S_max, size=(N_f, 1)).astype(np.float32)
t_f_np = rng.uniform(0.0, T, size=(N_f, 1)).astype(np.float32)

t_b_np = rng.uniform(0.0, T, size=(N_b, 1)).astype(np.float32)
S_left_np = np.zeros((N_b, 1), dtype=np.float32)
S_right_np = np.full((N_b, 1), S_max, dtype=np.float32)

S_T_np = rng.uniform(0.0, S_max, size=(N_T, 1)).astype(np.float32)
t_T_np = np.full((N_T, 1), T, dtype=np.float32)

S_f = tf.constant(S_f_np)
t_f = tf.constant(t_f_np)
t_b = tf.constant(t_b_np)
S_left = tf.constant(S_left_np)
S_right = tf.constant(S_right_np)
S_T = tf.constant(S_T_np)
t_T = tf.constant(t_T_np)

print("Interior points:", S_f.shape, t_f.shape)
print("Boundary points:", S_left.shape, S_right.shape)
print("Terminal points:", S_T.shape, t_T.shape)


### Why I generated collocation, boundary and terminal points

PINNs do not require a conventional labelled training dataset for the interior of the domain. Instead, collocation points are sampled in the $(S,t)$ domain and the governing PDE is enforced through its residual (Raissi et al., 2019).

Separate boundary and terminal samples are required because the Black–Scholes PDE alone does not uniquely determine the European call solution. The boundary conditions and maturity payoff supply the additional constraints needed to identify the correct solution.


## 7. Common evaluation grid


In [ ]:
S_common = np.linspace(0.0, S_max, 401)
t_common = np.zeros_like(S_common)

S_common_tf = tf.constant(
    S_common.reshape(-1, 1),
    dtype=tf.float32
)

t_common_tf = tf.constant(
    t_common.reshape(-1, 1),
    dtype=tf.float32
)

print("Common evaluation points:", len(S_common))


## 8. Error metric helper


In [ ]:
def calculate_metrics(prediction, exact):
    error = prediction - exact
    mse = np.mean(error**2)
    rmse = np.sqrt(mse)
    l2 = np.linalg.norm(error, ord=2)
    max_abs_error = np.max(np.abs(error))
    return error, mse, rmse, l2, max_abs_error


### Why several error metrics are used

No single error measure completely describes a numerical solution. MSE and RMSE summarise average discrepancy across the evaluation grid, the $L_2$ norm measures the total magnitude of the error vector, and maximum absolute error identifies the worst local deviation.

Using several metrics prevents the comparison from depending only on the price at $S=100$. The values produced are the author's own computations.


## 9. Train one PINN for a specified strike price


In [ ]:
def train_pinn_for_K(K_value):

    V_left = tf.zeros((N_b, 1), dtype=tf.float32)

    V_right = (
        S_max
        - K_value * tf.exp(-r * (T - t_b))
    )

    V_T = tf.maximum(
        S_T - K_value,
        0.0
    )

    U_left = V_left / K_value
    U_right = V_right / K_value
    U_T = V_T / K_value

    model = PINN()

    sample_S = tf.constant([[S0]], dtype=tf.float32)
    sample_t = tf.constant([[0.5]], dtype=tf.float32)
    _ = model_prediction(model, sample_S, sample_t)

    def pde_residual(S, t):
        with tf.GradientTape(persistent=True) as tape2:
            tape2.watch(S)
            tape2.watch(t)

            with tf.GradientTape(persistent=True) as tape1:
                tape1.watch(S)
                tape1.watch(t)
                u = model_prediction(model, S, t)

            u_S = tape1.gradient(u, S)
            u_t = tape1.gradient(u, t)

        u_SS = tape2.gradient(u_S, S)

        del tape1
        del tape2

        return (
            u_t
            + 0.5 * sigma**2 * S**2 * u_SS
            + r * S * u_S
            - r * u
        )

    def total_loss():
        residual = pde_residual(S_f, t_f)
        loss_pde = tf.reduce_mean(tf.square(residual))

        U_left_pred = model_prediction(model, S_left, t_b)
        U_right_pred = model_prediction(model, S_right, t_b)

        U_boundary_pred = tf.concat(
            [U_left_pred, U_right_pred],
            axis=0
        )

        U_boundary_true = tf.concat(
            [U_left, U_right],
            axis=0
        )

        loss_boundary = tf.reduce_mean(
            tf.square(U_boundary_pred - U_boundary_true)
        )

        U_T_pred = model_prediction(model, S_T, t_T)

        loss_terminal = tf.reduce_mean(
            tf.square(U_T_pred - U_T)
        )

        loss_total = (
            lambda_f * loss_pde
            + lambda_B * loss_boundary
            + lambda_T * loss_terminal
        )

        return (
            loss_total,
            loss_pde,
            loss_boundary,
            loss_terminal
        )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=LR_1
    )

    @tf.function(reduce_retracing=True)
    def train_step():
        with tf.GradientTape() as tape:
            (
                loss_total,
                loss_pde,
                loss_boundary,
                loss_terminal
            ) = total_loss()

        gradients = tape.gradient(
            loss_total,
            model.trainable_variables
        )

        pairs = [
            (g, v)
            for g, v in zip(gradients, model.trainable_variables)
            if g is not None
        ]

        optimizer.apply_gradients(pairs)

        return (
            loss_total,
            loss_pde,
            loss_boundary,
            loss_terminal
        )

    loss_history = []
    pde_history = []
    boundary_history = []
    terminal_history = []

    best_loss = np.inf
    best_epoch = 0
    best_weights = None

    start_time = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):

        if epoch == 3001:
            optimizer.learning_rate.assign(LR_2)

        if epoch == 7001:
            optimizer.learning_rate.assign(LR_3)

        (
            loss_total,
            loss_pde,
            loss_boundary,
            loss_terminal
        ) = train_step()

        current_total = float(loss_total.numpy())

        loss_history.append(current_total)
        pde_history.append(float(loss_pde.numpy()))
        boundary_history.append(float(loss_boundary.numpy()))
        terminal_history.append(float(loss_terminal.numpy()))

        if current_total < best_loss:
            best_loss = current_total
            best_epoch = epoch
            best_weights = model.get_weights()

        if epoch == 1 or epoch % 1000 == 0:
            print(
                f"K={K_value:.0f} | "
                f"Epoch {epoch:5d} | "
                f"LR={optimizer.learning_rate.numpy():.1e} | "
                f"Total={loss_total.numpy():.6e} | "
                f"PDE={loss_pde.numpy():.6e} | "
                f"B={loss_boundary.numpy():.6e} | "
                f"T={loss_terminal.numpy():.6e}"
            )

    training_time = time.perf_counter() - start_time
    model.set_weights(best_weights)

    return {
        "model": model,
        "best_epoch": best_epoch,
        "best_loss": best_loss,
        "training_time": training_time,
        "loss_history": loss_history,
        "pde_history": pde_history,
        "boundary_history": boundary_history,
        "terminal_history": terminal_history
    }


### Why I used automatic differentiation for the PDE residual

The PINN solution is constrained by the Black–Scholes PDE. TensorFlow automatic differentiation is therefore used to obtain

$$
u_t,\qquad u_S,\qquad u_{SS}.
$$

These derivatives are substituted directly into the differential equation to form the residual. Minimising the residual at collocation points is the central physics-informed mechanism that allows the network to learn the PDE solution without labelled interior solution values (Raissi et al., 2019; Karniadakis et al., 2021).

Nested gradient tapes are required because $u_{SS}$ is a second derivative.


## 10. Run all strike-price scenarios


In [ ]:
strike_results = []
trained_models = {}
training_histories = {}

for K_value in K_values:

    print("\n" + "=" * 70)
    print(f"STARTING STRIKE EXPERIMENT: K = {K_value:.0f}")
    print("=" * 70)

    V_exact = black_scholes_call(
        S_common,
        K_value,
        r,
        sigma,
        T,
        t=0.0
    )

    exact_price_S0 = black_scholes_call(
        S0,
        K_value,
        r,
        sigma,
        T,
        t=0.0
    )

    # FDM
    fdm_times = []

    for repeat in range(20):
        start = time.perf_counter()

        S_fdm, V_fdm = crank_nicolson_call(
            S_max=S_max,
            K=K_value,
            r=r,
            sigma=sigma,
            T=T,
            M=FDM_M,
            N=FDM_N
        )

        fdm_times.append(time.perf_counter() - start)

    fdm_runtime = np.median(fdm_times)

    V_fdm_common = np.interp(
        S_common,
        S_fdm,
        V_fdm
    )

    fdm_price_S0 = np.interp(
        S0,
        S_common,
        V_fdm_common
    )

    (
        fdm_error,
        fdm_mse,
        fdm_rmse,
        fdm_l2,
        fdm_max_error
    ) = calculate_metrics(
        V_fdm_common,
        V_exact
    )

    fdm_abs_price_error = abs(
        fdm_price_S0 - exact_price_S0
    )

    fdm_rel_price_error = (
        fdm_abs_price_error
        / exact_price_S0
        * 100.0
    )

    # PINN
    pinn_output = train_pinn_for_K(K_value)
    model = pinn_output["model"]

    trained_models[K_value] = model
    training_histories[K_value] = {
        "total": pinn_output["loss_history"],
        "pde": pinn_output["pde_history"],
        "boundary": pinn_output["boundary_history"],
        "terminal": pinn_output["terminal_history"]
    }

    _ = model_prediction(
        model,
        S_common_tf,
        t_common_tf
    ).numpy()

    inference_times = []

    for repeat in range(100):
        start = time.perf_counter()

        u_temp = model_prediction(
            model,
            S_common_tf,
            t_common_tf
        )

        _ = (K_value * u_temp).numpy()

        inference_times.append(time.perf_counter() - start)

    pinn_inference_runtime = np.median(
        inference_times
    )

    u_pinn = model_prediction(
        model,
        S_common_tf,
        t_common_tf
    ).numpy().reshape(-1)

    V_pinn = K_value * u_pinn

    pinn_price_S0 = np.interp(
        S0,
        S_common,
        V_pinn
    )

    (
        pinn_error,
        pinn_mse,
        pinn_rmse,
        pinn_l2,
        pinn_max_error
    ) = calculate_metrics(
        V_pinn,
        V_exact
    )

    pinn_abs_price_error = abs(
        pinn_price_S0 - exact_price_S0
    )

    pinn_rel_price_error = (
        pinn_abs_price_error
        / exact_price_S0
        * 100.0
    )

    strike_results.append({
        "Strike K": K_value,
        "Method": "Crank-Nicolson FDM",
        "Analytical Price": exact_price_S0,
        "Price at S=100": fdm_price_S0,
        "Absolute Price Error": fdm_abs_price_error,
        "Relative Price Error (%)": fdm_rel_price_error,
        "MSE": fdm_mse,
        "RMSE": fdm_rmse,
        "L2 Error": fdm_l2,
        "Maximum Abs Error": fdm_max_error,
        "Solve / Inference Runtime (s)": fdm_runtime,
        "Training Runtime (s)": np.nan,
        "Best Epoch": np.nan,
        "Best Total Loss": np.nan
    })

    strike_results.append({
        "Strike K": K_value,
        "Method": "Improved PINN",
        "Analytical Price": exact_price_S0,
        "Price at S=100": pinn_price_S0,
        "Absolute Price Error": pinn_abs_price_error,
        "Relative Price Error (%)": pinn_rel_price_error,
        "MSE": pinn_mse,
        "RMSE": pinn_rmse,
        "L2 Error": pinn_l2,
        "Maximum Abs Error": pinn_max_error,
        "Solve / Inference Runtime (s)": pinn_inference_runtime,
        "Training Runtime (s)": pinn_output["training_time"],
        "Best Epoch": pinn_output["best_epoch"],
        "Best Total Loss": pinn_output["best_loss"]
    })

    print(f"Completed K = {K_value:.0f}")

strike_df = pd.DataFrame(strike_results)
strike_df


### Why I used these model parameters and hyperparameters

The Black–Scholes parameters define the financial problem being solved. The baseline values are held fixed unless the notebook is explicitly performing a sensitivity study. This produces a controlled reference case against which numerical changes can be measured (Black & Scholes, 1973; Hull, 2018).

The PINN hyperparameters define the computational budget and network training configuration. Keeping these settings explicit is important for reproducibility and makes it possible to distinguish changes in model accuracy caused by the market parameter from changes caused by the numerical procedure.


## 11. Compact strike sensitivity table


In [ ]:
compact_strike_df = strike_df[
    [
        "Strike K",
        "Method",
        "Analytical Price",
        "Price at S=100",
        "Absolute Price Error",
        "Relative Price Error (%)",
        "MSE",
        "RMSE",
        "L2 Error",
        "Solve / Inference Runtime (s)",
        "Training Runtime (s)"
    ]
].copy()

compact_strike_df


## 12. Price sensitivity to strike


In [ ]:
fdm_rows = strike_df[
    strike_df["Method"] == "Crank-Nicolson FDM"
].sort_values("Strike K")

pinn_rows = strike_df[
    strike_df["Method"] == "Improved PINN"
].sort_values("Strike K")

plt.figure(figsize=(10, 6))

plt.plot(
    fdm_rows["Strike K"],
    fdm_rows["Analytical Price"],
    marker="o",
    label="Analytical Black-Scholes"
)

plt.plot(
    fdm_rows["Strike K"],
    fdm_rows["Price at S=100"],
    marker="o",
    linestyle="--",
    label="Crank-Nicolson FDM"
)

plt.plot(
    pinn_rows["Strike K"],
    pinn_rows["Price at S=100"],
    marker="o",
    linestyle=":",
    label="Improved PINN"
)

plt.xlabel("Strike Price (K)")
plt.ylabel("Call Option Price at S=100")
plt.title("Effect of Strike Price on European Call Price")
plt.legend()
plt.grid(True)
plt.show()


### Why I included this figure

The figure provides a visual comparison that complements the numerical error table. Curves that appear close on an option-price plot may still have materially different numerical errors, so the graphical results are interpreted together with MSE, RMSE, $L_2$ and maximum-error measures.



## 13. Absolute pricing error vs strike


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    fdm_rows["Strike K"],
    fdm_rows["Absolute Price Error"],
    marker="o",
    label="FDM"
)

plt.plot(
    pinn_rows["Strike K"],
    pinn_rows["Absolute Price Error"],
    marker="o",
    label="PINN"
)

plt.xlabel("Strike Price (K)")
plt.ylabel("Absolute Price Error")
plt.title("Absolute Pricing Error vs Strike Price")
plt.legend()
plt.grid(True)
plt.show()


### Why I included this figure

The figure provides a visual comparison that complements the numerical error table. Curves that appear close on an option-price plot may still have materially different numerical errors, so the graphical results are interpreted together with MSE, RMSE, $L_2$ and maximum-error measures.



## 14. MSE vs strike


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    fdm_rows["Strike K"],
    fdm_rows["MSE"],
    marker="o",
    label="FDM"
)

plt.plot(
    pinn_rows["Strike K"],
    pinn_rows["MSE"],
    marker="o",
    label="PINN"
)

plt.yscale("log")
plt.xlabel("Strike Price (K)")
plt.ylabel("MSE (log scale)")
plt.title("MSE Sensitivity to Strike Price")
plt.legend()
plt.grid(True)
plt.show()


### Why I included this figure

The figure provides a visual comparison that complements the numerical error table. Curves that appear close on an option-price plot may still have materially different numerical errors, so the graphical results are interpreted together with MSE, RMSE, $L_2$ and maximum-error measures.


## 15. PINN training runtime vs strike


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    pinn_rows["Strike K"],
    pinn_rows["Training Runtime (s)"],
    marker="o"
)

plt.xlabel("Strike Price (K)")
plt.ylabel("PINN Training Runtime (seconds)")
plt.title("PINN Training Runtime vs Strike Price")
plt.grid(True)
plt.show()


### Why I included this figure

The figure provides a visual comparison that complements the numerical error table. Curves that appear close on an option-price plot may still have materially different numerical errors, so the graphical results are interpreted together with MSE, RMSE, $L_2$ and maximum-error measures.


## 16. PINN convergence under different strike values


In [ ]:
plt.figure(figsize=(10, 6))

for K_value in K_values:
    plt.semilogy(
        training_histories[K_value]["total"],
        label=f"K = {K_value:.0f}"
    )

plt.axvline(3000, linestyle="--")
plt.axvline(7000, linestyle="--")

plt.xlabel("Epoch")
plt.ylabel("Total PINN Loss")
plt.title("PINN Convergence Under Different Strike Prices")
plt.legend()
plt.grid(True)
plt.show()


### Why I used a logarithmic scale for the loss graph

PINN losses often change by several orders of magnitude during training. A logarithmic vertical axis makes both the early large losses and the later small losses visible on the same figure. The graph is used to assess convergence and identify instability or spikes in the optimisation process. Matplotlib is used for the visualisation (Hunter, 2007).


## 17. Save numerical results


In [ ]:
strike_df.to_csv(
    "strike_sensitivity_results.csv",
    index=False
)

print("Saved: strike_sensitivity_results.csv")


## 18. Outputs used from this experiment

The main outputs I kept from this strike-price experiment were the full results table, the price-versus-strike graph, the absolute-error graph, the MSE graph, the PINN training-runtime graph and the convergence graph.




## Results obtained from the completed strike-price sensitivity study

|   Strike K | Method             |   Analytical Price |   Price at S=100 |   Absolute Price Error |   Relative Price Error (%) |         MSE |       RMSE |   L2 Error |   Solve / Inference Runtime (s) |   Training Runtime (s) |
|-----------:|:-------------------|-------------------:|-----------------:|-----------------------:|---------------------------:|------------:|-----------:|-----------:|--------------------------------:|-----------------------:|
|         80 | Crank-Nicolson FDM |           24.5888  |         24.5852  |             0.00360428 |                  0.0146582 | 7.2941e-06  | 0.00270076 |  0.0540826 |                      0.00363771 |                nan     |
|         80 | Improved PINN      |           24.5888  |         24.5827  |             0.00611755 |                  0.0248794 | 0.0082037   | 0.0905743  |  1.81375   |                      0.00553289 |               1026.86  |
|        100 | Crank-Nicolson FDM |           10.4506  |         10.4407  |             0.00987333 |                  0.0944763 | 5.83264e-06 | 0.00241509 |  0.0483621 |                      0.0035893  |                nan     |
|        100 | Improved PINN      |           10.4506  |         10.5498  |             0.099223   |                  0.94945   | 0.00483946  | 0.0695662  |  1.39306   |                      0.00549696 |               1012.81  |
|        120 | Crank-Nicolson FDM |            3.24748 |          3.24293 |             0.00454603 |                  0.139987  | 2.12962e-05 | 0.00461478 |  0.0924108 |                      0.0034883  |                nan     |
|        120 | Improved PINN      |            3.24748 |          3.29912 |             0.0516454  |                  1.59032   | 0.00263773  | 0.0513588  |  1.02846   |                      0.00827588 |                998.708 |

The call price decreased as the strike increased, consistent with standard European call behaviour (Black & Scholes, 1973; Hull, 2018). The FDM retained lower global error across all three strike values. The $K=80$, $K=100$ and $K=120$ cases also represent in-the-money, approximately at-the-money and out-of-the-money conditions at $S_0=100$, respectively (Hull, 2018).




## References — CPUT Harvard style

Black, F. & Scholes, M. 1973. The pricing of options and corporate liabilities. *Journal of Political Economy*, 81(3):637-654. DOI: 10.1086/260062.

Crank, J. & Nicolson, P. 1947. A practical method for numerical evaluation of solutions of partial differential equations of the heat-conduction type. *Proceedings of the Cambridge Philosophical Society*, 43(1):50-67. DOI: 10.1017/S0305004100023197.

Duffy, D.J. 2006. *Finite difference methods in financial engineering: A partial differential equation approach*. Chichester: John Wiley & Sons. DOI: 10.1002/9781118673447.

Harris, C.R., Millman, K.J., van der Walt, S.J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N.J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M.H., Brett, M., Haldane, A., del Río, J.F., Wiebe, M., Peterson, P., Gérard-Marchant, P., Sheppard, K., Reddy, T., Weckesser, W., Abbasi, H., Gohlke, C. & Oliphant, T.E. 2020. Array programming with NumPy. *Nature*, 585:357-362. DOI: 10.1038/s41586-020-2649-2.

Hull, J.C. 2018. *Options, futures, and other derivatives*. 10th ed. Harlow: Pearson.

Hunter, J.D. 2007. Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*, 9(3):90-95. DOI: 10.1109/MCSE.2007.55.

Karniadakis, G.E., Kevrekidis, I.G., Lu, L., Perdikaris, P., Wang, S. & Yang, L. 2021. Physics-informed machine learning. *Nature Reviews Physics*, 3(6):422-440. DOI: 10.1038/s42254-021-00314-5.

McKinney, W. 2010. Data structures for statistical computing in Python. In: van der Walt, S. & Millman, J. eds. *Proceedings of the 9th Python in Science Conference*. Austin, TX: SciPy, 56-61. DOI: 10.25080/Majora-92bf1922-00a.

Raissi, M., Perdikaris, P. & Karniadakis, G.E. 2019. Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. *Journal of Computational Physics*, 378:686-707. DOI: 10.1016/j.jcp.2018.10.045.

Virtanen, P., Gommers, R., Oliphant, T.E., Haberland, M., Reddy, T., Cournapeau, D., Burovski, E., Peterson, P., Weckesser, W., Bright, J., van der Walt, S.J., Brett, M., Wilson, J., Millman, K.J., Mayorov, N., Nelson, A.R.J., Jones, E., Kern, R., Larson, E., Carey, C.J., Polat, İ., Feng, Y., Moore, E.W., VanderPlas, J., Laxalde, D., Perktold, J., Cimrman, R., Henriksen, I., Quintero, E.A., Harris, C.R., Archibald, A.M., Ribeiro, A.H., Pedregosa, F., van Mulbregt, P. & SciPy 1.0 Contributors. 2020. SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods*, 17:261-272. DOI: 10.1038/s41592-019-0686-2.

Wang, S., Teng, Y. & Perdikaris, P. 2021. Understanding and mitigating gradient flow pathologies in physics-informed neural networks. *SIAM Journal on Scientific Computing*, 43(5):A3055-A3081. DOI: 10.1137/20M1318043.

